# L06 – Lösningar

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, precision_score, recall_score
from dlkit.seed import set_seed
from dlkit.data import fashion_mnist_loaders, FASHION_CLASSES
from dlkit.train import fit, predict

device = "cuda" if torch.cuda.is_available() else "cpu"
set_seed(42)
train_loader, val_loader, _ = fashion_mnist_loaders()
model = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(),
                      nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10)).to(device)
fit(model, train_loader, val_loader, nn.CrossEntropyLoss(),
    torch.optim.Adam(model.parameters(), lr=1e-3), device, epochs=10, verbose=False)
y_true, y_pred, probs = predict(model, val_loader, device)
cm = confusion_matrix(y_true, y_pred)

### Övning 1
- **Precision** för klass 6 är rätt gissade Shirt delat med *alla som gissades* Shirt, alltså kolumn 6.
- **Recall** är rätt gissade Shirt delat med *alla riktiga* Shirt, alltså rad 6.

In [ ]:
shirt_precision = cm[6, 6] / cm[:, 6].sum()
shirt_recall = cm[6, 6] / cm[6, :].sum()
assert np.isclose(shirt_precision, precision_score(y_true, y_pred, labels=[6], average="macro"))
assert np.isclose(shirt_recall, recall_score(y_true, y_pred, labels=[6], average="macro"))
print(f"Rätt! precision={shirt_precision:.3f} recall={shirt_recall:.3f}")

### Övning 2

In [ ]:
off = cm.copy()
np.fill_diagonal(off, 0)                       # bara felen kvar
flat = np.argsort(off, axis=None)[::-1][:3]    # index för de 3 största värdena
rows, cols = np.unravel_index(flat, off.shape)
top3 = [(FASHION_CLASSES[r], FASHION_CLASSES[c], int(off[r, c])) for r, c in zip(rows, cols)]
assert len(top3) == 3 and top3[0][2] >= top3[1][2] >= top3[2][2]
assert "Shirt" in (top3[0][0], top3[0][1])
print("Rätt!", top3)

### Övning 3 – förslag på svar
1. **Överanpassning.** Använd early stopping (behåll modellen från cirka epok 8), regularisering (weight decay, dropout), mer data eller augmentation.
2. **Underanpassning.** Train och val är lika och dåliga. Använd större modell (fler eller bredare lager), en bättre arkitektur för datan (till exempel CNN för bilder) eller högre learning rate, eller minska regulariseringen.
3. **Inte klar.** Träna fler epoker och se var kurvorna planar ut.